# Feature Engineering สำหรับ Dataset กลาง

Notebook นี้สร้าง model features จากผลการรวม/split ใน `common/output/` และเก็บ labels กับ metadata แยกไว้ ไม่ import ฟังก์ชันจากไฟล์ Python ภายนอก

หลักสำคัญ: เรียนรู้ scaler/encoder จาก `train` เท่านั้น แล้วใช้ตัวแปลงเดิมกับ `train_full`, `validation` และ `test`; การ fit ไม่ใช้ `y`

## 1. เตรียม path และการตรวจตาราง

หา project root อัตโนมัติ แล้วกำหนดตำแหน่ง input/output รวมถึงตรวจว่า `record_id` ของสามตารางตรงกัน.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def find_project_root(start=None):
    """หาโฟลเดอร์โปรเจกต์จากตำแหน่งที่เปิด notebook โดยไม่ล็อก path เครื่องใดเครื่องหนึ่ง."""
    origin = Path(start or Path.cwd()).resolve()
    for candidate in (origin, *origin.parents):
        has_pipeline = (candidate / "data-pipeline").is_dir()
        has_project_marker = (candidate / ".git").exists() or (candidate / "README.md").is_file()
        if has_pipeline and has_project_marker:
            return candidate
    raise FileNotFoundError(f"หาโฟลเดอร์โปรเจกต์จากตำแหน่งนี้ไม่พบ: {origin}")


PROJECT_ROOT = find_project_root()
COMMON_ROOT = PROJECT_ROOT / "data-pipeline" / "common"
BUILD_ROOT = COMMON_ROOT / "output"
OUTPUT_ROOT = BUILD_ROOT / "feature_engineering"
SPLITS = ("train", "train_full", "validation", "test")
KEY = "record_id"
FORBIDDEN_FEATURE_COLUMNS = {
    "label", "label_raw", "attack_category", "label_binary", "is_attack",
    "source_dataset", "orig_row_id", "capture_time", "split_group",
    "sample_weight", "dedup_flag", "data_split", "difficulty_level",
}


def validate_aligned_tables(features, labels, metadata):
    """ตรวจการเชื่อม features/labels/metadata และป้องกันข้อมูลเป้าหมายรั่วเข้า X."""
    tables = {"features": features, "labels": labels, "metadata": metadata}
    for table_name, frame in tables.items():
        if KEY not in frame.columns:
            raise ValueError(f"ตาราง {table_name} ไม่มีคอลัมน์ {KEY!r}")
        if frame[KEY].isna().any() or frame[KEY].astype(str).eq("").any():
            raise ValueError(f"{table_name}.{KEY} มีค่าว่าง")
        if frame[KEY].duplicated().any():
            examples = frame.loc[frame[KEY].duplicated(), KEY].head(3).tolist()
            raise ValueError(f"{table_name}.{KEY} ซ้ำ ตัวอย่าง: {examples}")
    reference_ids = features[KEY].astype(str).reset_index(drop=True)
    for table_name, frame in (("labels", labels), ("metadata", metadata)):
        if not reference_ids.equals(frame[KEY].astype(str).reset_index(drop=True)):
            raise ValueError(f"ลำดับ record_id ใน features และ {table_name} ไม่ตรงกัน")
    leaked_columns = FORBIDDEN_FEATURE_COLUMNS.intersection(features.columns)
    if leaked_columns:
        raise ValueError(f"พบ label/metadata ปะปนใน features: {sorted(leaked_columns)}")
    if set(labels.columns) == {KEY}:
        raise ValueError("ตาราง labels ไม่มีคอลัมน์ label")
    return {name: len(frame) for name, frame in tables.items()}


print(f"โฟลเดอร์โปรเจกต์: {PROJECT_ROOT}")
print(f"ข้อมูลกลางที่อ่าน: {BUILD_ROOT}")
print(f"ผล Feature Engineering: {OUTPUT_ROOT}")


## 2. คำนวณ feature จากค่าพื้นฐาน

ใช้ `log1p` ลดผลของค่าที่มีหางยาว และสร้างปริมาณรวม อัตราส่วน bytes สองทิศทาง และ bytes ต่อวินาทีเมื่อข้อมูลต้นทางพร้อม สูตรคำนวณจากข้อมูลในแต่ละ record ไม่ใช้ label.

In [ ]:
def _derive(frame: pd.DataFrame, usable_core_features: list[str]) -> pd.DataFrame:
    """สร้าง feature แบบ log จาก bytes/duration โดยไม่แก้ input dataframe."""
    output = pd.DataFrame(index=frame.index)
    if "duration" in usable_core_features:
        output["log_duration"] = np.log1p(frame["duration"].astype(float))
    if "bytes_forward" in usable_core_features:
        output["log_bytes_forward"] = np.log1p(frame["bytes_forward"].astype(float))
    if "bytes_backward" in usable_core_features:
        output["log_bytes_backward"] = np.log1p(frame["bytes_backward"].astype(float))
    if {"bytes_forward", "bytes_backward"}.issubset(usable_core_features):
        forward = frame["bytes_forward"].astype(float)
        backward = frame["bytes_backward"].astype(float)
        output["log_total_bytes"] = np.log1p(forward + backward)
        output["log_bytes_ratio"] = np.log1p(forward / (backward + 1.0))
        if "duration" in usable_core_features:
            duration = frame["duration"].astype(float)
            output["log_bytes_per_second"] = np.log1p((forward + backward) / np.maximum(duration, 1e-6))
    return output

## 3. ตรวจความพร้อม เลือก feature และอ่าน split

อ่านรายงานความเข้ากันได้จากขั้น Build ก่อนเลือก core features หากมี dataset เดียวพร้อม จะระบุขอบเขตเป็น `source_local_diagnostic` ไม่อ้างว่าเป็น feature ที่ยืนยันร่วมกันข้ามสอง dataset.

In [ ]:
def run_feature_engineering() -> dict:
    """สร้าง model input โดยเรียนรู้ scaler/encoder จาก Train เท่านั้น."""
    compatibility_path = BUILD_ROOT / "reports" / "feature_compatibility.json"
    if not compatibility_path.is_file():
        raise FileNotFoundError("Run common/build_dataset.ipynb before central feature engineering.")
    compatibility_checks = json.loads(compatibility_path.read_text(encoding="utf-8"))
    usable_core_features = [
        row["feature"]
        for row in compatibility_checks
        if row.get("usable_in_current_build", row["eligible_for_joint_profile"])
    ]
    if not usable_core_features:
        raise ValueError("No core feature is complete in the dataset(s) available for this build.")

    readiness_path = BUILD_ROOT / "reports" / "dataset_readiness.csv"
    if readiness_path.is_file():
        readiness = pd.read_csv(readiness_path)
        joint_profile_ready = bool(len(readiness)) and bool(readiness["ready"].astype(bool).all())
    else:
        joint_profile_ready = all(
            row.get("eligible_for_joint_profile", False) for row in compatibility_checks
        )
    profile_scope = "joint" if joint_profile_ready else "source_local_diagnostic"

    def read_split(name: str):
        folder = BUILD_ROOT / "splits" / name
        features = pd.read_csv(folder / "features.csv")
        labels = pd.read_csv(folder / "labels.csv", dtype={"label_raw": "string"})
        metadata = pd.read_csv(folder / "metadata.csv")
        validate_aligned_tables(features, labels, metadata)
        return features, labels, metadata

    splits = {name: read_split(name) for name in SPLITS}
    # Fit preprocessing จาก Train ที่สุ่มให้สมดุลแหล่งข้อมูลเท่านั้น
    numeric_train = _derive(splits["train"][0], usable_core_features)
    if numeric_train.empty:
        raise ValueError("The shared profile has no numeric features to train a preprocessing transformer.")
    for name, (features, _, _) in splits.items():
        if name == "train_full":
            continue
        if features[usable_core_features].isna().any().any():
            missing = features[usable_core_features].isna().sum()
            raise ValueError(
                f"{name} contains null values in selected shared features: {missing[missing > 0].to_dict()}. "
                "The baseline profile will not silently impute or convert unavailable values to zero."
            )
    if numeric_train.isna().any().any() or not np.isfinite(numeric_train.to_numpy(dtype=float)).all():
        raise ValueError("Derived training features contain missing or infinite values.")

    # StandardScaler และ OneHotEncoder เรียนรู้ค่าจาก Train แล้ว reuse กับ split อื่น
    scaler = StandardScaler()
    scaler.fit(numeric_train)  # fit บน Train; ยังไม่ต้องสร้าง transformed array ที่ไม่ได้ใช้
    encoder = None
    use_protocol = "protocol" in usable_core_features
    if use_protocol:
        encoder = OneHotEncoder(
            categories=[["tcp", "udp", "icmp", "other"]],
            handle_unknown="ignore",
            sparse_output=False,
        )
        train_protocol = splits["train"][0][["protocol"]].astype("string")
        encoder.fit(train_protocol)
    numeric_names = numeric_train.columns.tolist()
    protocol_names = encoder.get_feature_names_out(["protocol"]).tolist() if encoder is not None else []
    output_names = numeric_names + protocol_names

    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    import joblib
    preprocessor_path = OUTPUT_ROOT / "preprocessor.joblib"
    joblib.dump({"scaler": scaler, "encoder": encoder,
                 "usable_core_features": usable_core_features,
                 "numeric_feature_names": numeric_names, "model_feature_names": output_names,
                 "fit_split": "train"}, preprocessor_path)

    # แปลงและบันทึกแต่ละ split; label/metadata ไม่ถูกรวมเข้า model features
    reports = []
    for name, (features, labels, metadata) in splits.items():
        derived = _derive(features, usable_core_features)
        if derived.isna().any().any() or not np.isfinite(derived.to_numpy(dtype=float)).all():
            raise ValueError(f"{name} has missing/infinite engineered values; no records were silently dropped.")
        numeric_scaled = scaler.transform(derived[numeric_names])
        if encoder is not None:
            protocol_encoded = encoder.transform(features[["protocol"]].astype("string"))
            matrix = np.concatenate([numeric_scaled, protocol_encoded], axis=1)
        else:
            matrix = numeric_scaled
        model_features = pd.DataFrame(matrix, columns=output_names)
        model_features.insert(0, "record_id", features["record_id"].astype(str).to_numpy())
        target_output = labels.copy()
        metadata_output = metadata.copy()
        validate_aligned_tables(model_features, target_output, metadata_output)
        folder = OUTPUT_ROOT / name
        folder.mkdir(parents=True, exist_ok=True)
        model_features.to_csv(folder / "features.csv", index=False, encoding="utf-8")
        target_output.to_csv(folder / "labels.csv", index=False, encoding="utf-8")
        metadata_output.to_csv(folder / "metadata.csv", index=False, encoding="utf-8")
        reports.append({"split": name, "profile_scope": profile_scope,
                        "rows": len(model_features), "model_features": len(output_names),
                        "source_datasets": ",".join(
                            sorted(metadata["source_dataset"].unique())
                        )})

    schema = {"feature_names": output_names, "numeric_features": numeric_names,
              "categorical_features": protocol_names,
              "core_features_used": usable_core_features,
              "profile_scope": profile_scope, "joint_profile_ready": joint_profile_ready,
              "source_datasets": sorted(splits["train"][2]["source_dataset"].astype(str).unique()),
              "fit_on": "train", "fit_records": len(splits["train"][0]),
              "source_dataset_is_not_a_model_feature": True,
              "metadata_and_targets_are_saved_separately": True,
              "note": "record_id is included in the feature table only for joins; drop it before passing X to a model."}
    (OUTPUT_ROOT / "feature_schema.json").write_text(json.dumps(schema, indent=2, ensure_ascii=False), encoding="utf-8")
    pd.DataFrame(reports).to_csv(OUTPUT_ROOT / "feature_engineering_report.csv", index=False)
    return {"output_directory": str(OUTPUT_ROOT), "preprocessor": str(preprocessor_path),
            "core_features_used": usable_core_features,
            "engineered_features": output_names,
            "profile_scope": profile_scope, "joint_profile_ready": joint_profile_ready,
            "fit_rows": len(splits["train"][0]), "splits": reports}

## 4. สร้างผลลัพธ์

รันหลังจากทำ Build Dataset กลางแล้ว ระบบจะ fit preprocessing บน Train และส่งผลลัพธ์แยกตาม split ไปที่ `common/output/feature_engineering/` พร้อม schema และรายงานจำนวนแถว/feature.

In [ ]:
feature_engineering_summary = run_feature_engineering()
display(pd.DataFrame(feature_engineering_summary["splits"]))
print("ขอบเขตโปรไฟล์:", feature_engineering_summary["profile_scope"])
print("Core features:", feature_engineering_summary["core_features_used"])
print("ไฟล์ผลลัพธ์:", feature_engineering_summary["output_directory"])